<a href="https://colab.research.google.com/github/rebeca07-pedrozo/SINAI/blob/main/sinai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalacion de librerias

In [ ]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-spa
!pip install -q pymupdf
!pip install -q pdfplumber
!pip install -q pytesseract
!pip install -q easyocr
!pip install -q pandas
!pip install -q gspread google-auth

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## Verificacion de instalacion

In [ ]:
import fitz
import pdfplumber
import pytesseract
import easyocr
import pandas as pd
import gspread
print("PyMuPDF (fitz):", fitz.__doc__.split('\n')[0] if fitz.__doc__ else "OK")
print("pdfplumber: OK")
print("pytesseract: OK, versión Tesseract detectada", pytesseract.get_tesseract_version())
idiomas = pytesseract.get_languages()
assert 'spa' in idiomas, "El paquete de idioma español no se instaló correctamente."

PyMuPDF (fitz): PyMuPDF 1.28.0: Python bindings for the MuPDF 1.29.0 library.
pdfplumber: OK
pytesseract: OK, versión Tesseract detectada 4.1.1


# Alistamiento del entorno

## Librerias y Drive API

In [ ]:
import os
import logging
from pathlib import Path
from datetime import datetime
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Rutas

In [ ]:
DRIVE_BASE_PATH = Path('/content/drive/MyDrive/DAVIVIENDA')
PATHS = {
    "input":  DRIVE_BASE_PATH / "input_pdfs",
    "output": DRIVE_BASE_PATH / "output",
    "logs":   DRIVE_BASE_PATH / "logs",
    "temp":   DRIVE_BASE_PATH / "temp",
}

for nombre, ruta in PATHS.items():
    ruta.mkdir(parents=True, exist_ok=True)
    print(f"Carpeta '{nombre}' lista en: {ruta}")


Carpeta 'input' lista en: /content/drive/MyDrive/DAVIVIENDA/input_pdfs
Carpeta 'output' lista en: /content/drive/MyDrive/DAVIVIENDA/output
Carpeta 'logs' lista en: /content/drive/MyDrive/DAVIVIENDA/logs
Carpeta 'temp' lista en: /content/drive/MyDrive/DAVIVIENDA/temp


## Pipeline

In [ ]:
CONFIG = {
    "ocr_lang": "spa",
    "render_dpi": 300,
    "ocr_confidence_threshold": 60,
    "min_chars_texto_digital": 100,
    "encoding": "utf-8",
}

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = PATHS["logs"] / f"pipeline_{timestamp}.log"

file_handler = logging.FileHandler(log_file, encoding="utf-8")
stream_handler = logging.StreamHandler()

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)

logger = logging.getLogger("pipeline_normativas")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.addHandler(file_handler)
logger.addHandler(stream_handler)


class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger.removeHandler(file_handler)
file_handler = FlushFileHandler(log_file, encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

logger.info("Entorno configurado correctamente.")
logger.info(f"CONFIG cargado: {CONFIG}")
print(f"\nLog de esta sesión se está guardando en: {log_file}")

2026-07-28 18:27:13,304 | INFO | Entorno configurado correctamente.
INFO:pipeline_normativas:Entorno configurado correctamente.
2026-07-28 18:27:13,307 | INFO | CONFIG cargado: {'ocr_lang': 'spa', 'render_dpi': 300, 'ocr_confidence_threshold': 60, 'min_chars_texto_digital': 100, 'encoding': 'utf-8'}
INFO:pipeline_normativas:CONFIG cargado: {'ocr_lang': 'spa', 'render_dpi': 300, 'ocr_confidence_threshold': 60, 'min_chars_texto_digital': 100, 'encoding': 'utf-8'}



Log de esta sesión se está guardando en: /content/drive/MyDrive/DAVIVIENDA/logs/pipeline_20260728_182713.log


# Carga de archivos

## Escaneo y metadata

In [ ]:
import fitz

def listar_y_validar_pdfs(carpeta_input: Path) -> list[dict]:
    archivos_pdf = sorted(carpeta_input.glob("*.pdf"))
    logger.info(f"Se encontraron {len(archivos_pdf)} archivo(s) .pdf en {carpeta_input}")

    if not archivos_pdf:
        logger.warning(
            f"No hay PDFs en {carpeta_input}. "
            f"Carga nuevamente los PDFs a Drive."
        )
        return []

    resultado = []
    for ruta_pdf in archivos_pdf:
        try:
            doc = fitz.open(ruta_pdf)

            if doc.is_encrypted:
                logger.warning(f"'{ruta_pdf.name}' El archivo tiene contraseña.")
                doc.close()
                continue

            info = {
                "nombre": ruta_pdf.name,
                "ruta": ruta_pdf,
                "tamano_kb": round(ruta_pdf.stat().st_size / 1024, 1),
                "num_paginas": doc.page_count,
            }
            resultado.append(info)
            logger.info(f"Validado OK: {info['nombre']} ({info['num_paginas']} páginas, {info['tamano_kb']} KB)")
            doc.close()

        except Exception as e:
            logger.error(f"'{ruta_pdf.name}' no se pudo abrir, posiblemente corrupto. Detalle: {e}")
            continue

    return resultado

## Resumen

In [ ]:
pdfs_disponibles = listar_y_validar_pdfs(PATHS["input"])

if pdfs_disponibles:
    df_resumen = pd.DataFrame(pdfs_disponibles)[["nombre", "num_paginas", "tamano_kb"]]
    print("\nArchivos listos para procesar:")
    display(df_resumen)
else:
    print("\nNo hay archivos para procesar todavía.")

2026-07-28 18:27:25,509 | INFO | Se encontraron 1 archivo(s) .pdf en /content/drive/MyDrive/DAVIVIENDA/input_pdfs
INFO:pipeline_normativas:Se encontraron 1 archivo(s) .pdf en /content/drive/MyDrive/DAVIVIENDA/input_pdfs
2026-07-28 18:27:25,523 | INFO | Validado OK: NuevoDocumento-2019-06-26-07.57.04.pdf (7 páginas, 2483.8 KB)
INFO:pipeline_normativas:Validado OK: NuevoDocumento-2019-06-26-07.57.04.pdf (7 páginas, 2483.8 KB)



Archivos listos para procesar:


,nombre,num_paginas,tamano_kb
0,NuevoDocumento-2019-06-26-07.57.04.pdf,7,2483.8


# Deteccion del tipo de PDF

In [ ]:
def clasificar_paginas(ruta_pdf: Path) -> list[dict]:
    doc = fitz.open(ruta_pdf)
    total_paginas = doc.page_count
    clasificacion = []

    for num_pagina in range(total_paginas):
        page = doc[num_pagina]

        texto_extraido = page.get_text().strip()
        num_caracteres = len(texto_extraido)
        tiene_imagenes = len(page.get_images()) > 0

        if num_caracteres >= CONFIG["min_chars_texto_digital"]:
            tipo = "texto_digital"
        elif tiene_imagenes:
            tipo = "imagen_escaneada"
        else:
            tipo = "vacia"

        clasificacion.append({
            "archivo": ruta_pdf.name,
            "pagina": num_pagina + 1,
            "tipo": tipo,
            "num_caracteres_detectados": num_caracteres,
            "tiene_imagenes": tiene_imagenes,
        })

    doc.close()
    logger.info(f"'{ruta_pdf.name}': clasificación de {total_paginas} página(s) completada.")
    return clasificacion

clasificacion_completa = []
for pdf_info in pdfs_disponibles:
    clasificacion_completa.extend(clasificar_paginas(pdf_info["ruta"]))

df_clasificacion = pd.DataFrame(clasificacion_completa)

print("Clasificación detallada por página:")
display(df_clasificacion)

print("\nResumen por tipo de página:")
display(df_clasificacion["tipo"].value_counts())

2026-07-28 18:27:33,382 | INFO | 'NuevoDocumento-2019-06-26-07.57.04.pdf': clasificación de 7 página(s) completada.
INFO:pipeline_normativas:'NuevoDocumento-2019-06-26-07.57.04.pdf': clasificación de 7 página(s) completada.


Clasificación detallada por página:


,archivo,pagina,tipo,num_caracteres_detectados,tiene_imagenes
0,NuevoDocumento-2019-06-26-07.57.04.pdf,1,imagen_escaneada,21,True
1,NuevoDocumento-2019-06-26-07.57.04.pdf,2,imagen_escaneada,21,True
2,NuevoDocumento-2019-06-26-07.57.04.pdf,3,imagen_escaneada,21,True
3,NuevoDocumento-2019-06-26-07.57.04.pdf,4,imagen_escaneada,21,True
4,NuevoDocumento-2019-06-26-07.57.04.pdf,5,imagen_escaneada,21,True
5,NuevoDocumento-2019-06-26-07.57.04.pdf,6,imagen_escaneada,21,True
6,NuevoDocumento-2019-06-26-07.57.04.pdf,7,imagen_escaneada,21,True



Resumen por tipo de página:


,count
tipo,
imagen_escaneada,7


# Extraccion del texto

## Lector EasyOCR

In [ ]:

import numpy as np
from PIL import Image
import io
lector_easyocr = easyocr.Reader(['es'], gpu=False)
def extraer_texto_digital(page) -> str:
    return page.get_text().strip()
def renderizar_pagina_a_imagen(page, dpi: int) -> Image.Image:
    pixmap = page.get_pixmap(dpi=dpi)
    img_bytes = pixmap.tobytes("png")
    return Image.open(io.BytesIO(img_bytes))
def ocr_con_tesseract(imagen: Image.Image, idioma: str) -> tuple[str, float]:
    datos = pytesseract.image_to_data(imagen, lang=idioma, output_type=pytesseract.Output.DICT)

    lineas = {}
    confianzas = []

    for i, texto in enumerate(datos["text"]):
        texto = texto.strip()
        conf = int(datos["conf"][i]) if str(datos["conf"][i]).lstrip("-").isdigit() else -1

        if texto and conf >= 0:
            clave_linea = (datos["block_num"][i], datos["par_num"][i], datos["line_num"][i])
            lineas.setdefault(clave_linea, []).append(texto)
            confianzas.append(conf)

    lineas_ordenadas = sorted(lineas.keys())
    texto_final = "\n".join(" ".join(lineas[clave]) for clave in lineas_ordenadas)

    confianza_promedio = sum(confianzas) / len(confianzas) if confianzas else 0.0
    return texto_final, confianza_promedio
def ocr_con_easyocr(imagen: Image.Image) -> str:
    resultado = lector_easyocr.readtext(np.array(imagen), detail=0)
    return " ".join(resultado)
def extraer_texto_pagina(page, tipo: str, ruta_pdf_nombre: str, num_pagina: int) -> dict:
    if tipo == "texto_digital":
        texto = extraer_texto_digital(page)
        return {"texto": texto, "metodo": "extraccion_directa", "confianza_ocr": None}

    elif tipo == "imagen_escaneada":
        imagen = renderizar_pagina_a_imagen(page, dpi=CONFIG["render_dpi"])
        ruta_temp = PATHS["temp"] / f"{ruta_pdf_nombre}_pag{num_pagina}.png"
        imagen.save(ruta_temp)

        texto_tesseract, confianza = ocr_con_tesseract(imagen, idioma=CONFIG["ocr_lang"])

        if confianza < CONFIG["ocr_confidence_threshold"]:
            logger.warning(
                f"'{ruta_pdf_nombre}' pág.{num_pagina}: confianza Tesseract baja ({confianza:.1f}), "
                f"reintentando con EasyOCR."
            )
            texto_easyocr = ocr_con_easyocr(imagen)
            return {"texto": texto_easyocr, "metodo": "ocr_easyocr_fallback", "confianza_ocr": confianza}

        return {"texto": texto_tesseract, "metodo": "ocr_tesseract", "confianza_ocr": confianza}

    else:
        return {"texto": "", "metodo": "pagina_vacia", "confianza_ocr": None}

## Procesamiento de paginas

In [ ]:
resultados_extraccion = []
for pdf_info in pdfs_disponibles:
    doc = fitz.open(pdf_info["ruta"])
    filas_pdf = df_clasificacion[df_clasificacion["archivo"] == pdf_info["nombre"]]

    for _, fila in filas_pdf.iterrows():
        num_pagina = fila["pagina"]
        page = doc[num_pagina - 1]
        extraccion = extraer_texto_pagina(page, fila["tipo"], pdf_info["nombre"], num_pagina)
        resultados_extraccion.append({
            "archivo": pdf_info["nombre"],
            "pagina": num_pagina,
            "tipo_pagina": fila["tipo"],
            "metodo_extraccion": extraccion["metodo"],
            "confianza_ocr": extraccion["confianza_ocr"],
            "texto_extraido": extraccion["texto"],
            "num_caracteres": len(extraccion["texto"]),
        })
        logger.info(
            f"'{pdf_info['nombre']}' pág.{num_pagina}: método={extraccion['metodo']}, "
            f"{len(extraccion['texto'])} caracteres extraídos."
        )
    doc.close()
df_extraccion = pd.DataFrame(resultados_extraccion)

2026-07-28 18:28:03,299 | INFO | 'NuevoDocumento-2019-06-26-07.57.04.pdf' pág.1: método=ocr_tesseract, 2755 caracteres extraídos.
INFO:pipeline_normativas:'NuevoDocumento-2019-06-26-07.57.04.pdf' pág.1: método=ocr_tesseract, 2755 caracteres extraídos.
2026-07-28 18:28:15,780 | INFO | 'NuevoDocumento-2019-06-26-07.57.04.pdf' pág.2: método=ocr_tesseract, 3237 caracteres extraídos.
INFO:pipeline_normativas:'NuevoDocumento-2019-06-26-07.57.04.pdf' pág.2: método=ocr_tesseract, 3237 caracteres extraídos.
2026-07-28 18:28:28,253 | INFO | 'NuevoDocumento-2019-06-26-07.57.04.pdf' pág.3: método=ocr_tesseract, 4180 caracteres extraídos.
INFO:pipeline_normativas:'NuevoDocumento-2019-06-26-07.57.04.pdf' pág.3: método=ocr_tesseract, 4180 caracteres extraídos.
2026-07-28 18:28:42,981 | INFO | 'NuevoDocumento-2019-06-26-07.57.04.pdf' pág.4: método=ocr_tesseract, 3900 caracteres extraídos.
INFO:pipeline_normativas:'NuevoDocumento-2019-06-26-07.57.04.pdf' pág.4: método=ocr_tesseract, 3900 caracteres ext

In [ ]:
print("Resumen de extracción por página:")
display(df_extraccion[["archivo", "pagina", "metodo_extraccion", "confianza_ocr", "num_caracteres"]])

Resumen de extracción por página:


,archivo,pagina,metodo_extraccion,confianza_ocr,num_caracteres
0,NuevoDocumento-2019-06-26-07.57.04.pdf,1,ocr_tesseract,87.738041,2755
1,NuevoDocumento-2019-06-26-07.57.04.pdf,2,ocr_tesseract,92.067061,3237
2,NuevoDocumento-2019-06-26-07.57.04.pdf,3,ocr_tesseract,92.347626,4180
3,NuevoDocumento-2019-06-26-07.57.04.pdf,4,ocr_tesseract,91.636364,3900
4,NuevoDocumento-2019-06-26-07.57.04.pdf,5,ocr_tesseract,89.497436,3673
5,NuevoDocumento-2019-06-26-07.57.04.pdf,6,ocr_tesseract,89.642036,3669
6,NuevoDocumento-2019-06-26-07.57.04.pdf,7,ocr_tesseract,91.148225,2967


# Limpieza del texto

## Parametros

In [ ]:
import re
from difflib import SequenceMatcher
from collections import Counter
CONFIG["similitud_boilerplate"] = 0.75
CONFIG["frecuencia_boilerplate"] = 0.4
CONFIG["min_ratio_alfabetico_ruido"] = 0.5
CONFIG["min_longitud_para_filtro_ruido"] = 3

## Funciones de limpieza

In [ ]:
def normalizar_espacios(texto: str) -> str:
    lineas = texto.split("\n")
    lineas_normalizadas = [re.sub(r"[ \t]+", " ", linea).strip() for linea in lineas]
    return "\n".join(linea for linea in lineas_normalizadas if linea)


def normalizar_para_comparacion(texto: str) -> str:
    return re.sub(r"\d+", "", texto.lower()).strip()


def similares(a: str, b: str, umbral: float) -> bool:
    a_norm = normalizar_para_comparacion(a)
    b_norm = normalizar_para_comparacion(b)
    if len(a_norm) < 4 or len(b_norm) < 4:
        return False
    return SequenceMatcher(None, a_norm, b_norm).ratio() >= umbral


def detectar_lineas_boilerplate(lineas_por_pagina: list[list[str]]) -> set[str]:
    todas_las_lineas = [linea for pagina in lineas_por_pagina for linea in pagina]
    num_paginas = len(lineas_por_pagina)
    lineas_unicas = list(set(todas_las_lineas))
    boilerplate = set()

    for linea_candidata in lineas_unicas:
        if len(linea_candidata) < CONFIG["min_longitud_para_filtro_ruido"]:
            continue
        paginas_donde_aparece_similar = 0
        for pagina in lineas_por_pagina:
            if any(similares(linea_candidata, l, CONFIG["similitud_boilerplate"]) for l in pagina):
                paginas_donde_aparece_similar += 1
        proporcion = paginas_donde_aparece_similar / num_paginas
        if proporcion >= CONFIG["frecuencia_boilerplate"]:
            boilerplate.add(linea_candidata)

    return boilerplate


def es_linea_ruido(linea: str) -> bool:
    if len(linea) < CONFIG["min_longitud_para_filtro_ruido"]:
        return False
    alfanumericos = sum(1 for c in linea if c.isalnum())
    ratio_alfanumerico = alfanumericos / len(linea)
    return ratio_alfanumerico < CONFIG["min_ratio_alfabetico_ruido"]


def eliminar_marca_escaner(texto: str) -> str:
    patron = re.compile(r"^\s*scanned\s+by\s+camscanner\s*$", re.IGNORECASE)
    lineas = [l for l in texto.split("\n") if not patron.match(l)]
    return "\n".join(lineas)


PALABRAS_CORTAS_VALIDAS = {"a", "o", "y", "e", "u", "el", "la", "lo", "de", "en", "un", "su", "al", "es", "le", "se", "no", "si", "sí"}

def limpiar_tokens_garbage(linea: str) -> str:
    simbolos_inicio = ("=", "'", "+", ">", "|", "°", "\u201c", "\u201d")
    tokens_limpios = []
    for token in linea.split(" "):
        if not token:
            continue
        if token.startswith(simbolos_inicio):
            continue
        token_sin_puntuacion = token.strip(".,;:¿?¡!\"'")
        es_letra_suelta_invalida = (
            len(token_sin_puntuacion) == 1
            and token_sin_puntuacion.lower() not in PALABRAS_CORTAS_VALIDAS
            and not token_sin_puntuacion.isdigit()
        )
        if es_letra_suelta_invalida:
            continue
        tokens_limpios.append(token)
    return " ".join(tokens_limpios)


def limpiar_documento(df_paginas_doc: pd.DataFrame) -> pd.DataFrame:
    textos_normalizados = [normalizar_espacios(t) for t in df_paginas_doc["texto_extraido"]]
    lineas_por_pagina = [t.split("\n") for t in textos_normalizados]

    boilerplate = detectar_lineas_boilerplate(lineas_por_pagina)
    logger.info(f"Líneas de boilerplate detectadas ({len(boilerplate)}): {list(boilerplate)[:5]}...")

    textos_limpios = []
    for lineas in lineas_por_pagina:
        lineas_finales = []
        for linea in lineas:
            linea = limpiar_tokens_garbage(linea)
            if not linea:
                continue
            if any(similares(linea, b, CONFIG["similitud_boilerplate"]) for b in boilerplate):
                continue
            if es_linea_ruido(linea):
                continue
            lineas_finales.append(linea)

        texto_pagina_limpio = eliminar_marca_escaner("\n".join(lineas_finales))
        textos_limpios.append(texto_pagina_limpio.strip())

    df_resultado = df_paginas_doc.copy()
    df_resultado["texto_limpio"] = textos_limpios
    df_resultado["num_caracteres_limpio"] = df_resultado["texto_limpio"].str.len()
    return df_resultado

In [ ]:
dfs_limpios = []
for pdf_info in pdfs_disponibles:
    df_pdf = df_extraccion[df_extraccion["archivo"] == pdf_info["nombre"]].reset_index(drop=True)
    dfs_limpios.append(limpiar_documento(df_pdf))

df_limpio = pd.concat(dfs_limpios, ignore_index=True)
logger.info(f"Limpieza aplicada a {len(df_limpio)} página(s) en total.")

2026-07-28 18:29:37,164 | INFO | Líneas de boilerplate detectadas (31): ['Información Linea 145', 'Cerrera 30 Ho 75-90', 'Información Linea 195 206', ', DE BOGOTÁ Dc.', 'Carrera 20 No 25-90']...
INFO:pipeline_normativas:Líneas de boilerplate detectadas (31): ['Información Linea 145', 'Cerrera 30 Ho 75-90', 'Información Linea 195 206', ', DE BOGOTÁ Dc.', 'Carrera 20 No 25-90']...
2026-07-28 18:29:38,247 | INFO | Limpieza aplicada a 7 página(s) en total.
INFO:pipeline_normativas:Limpieza aplicada a 7 página(s) en total.


# Estructuración de la información

##Parametros de la estructuración

In [ ]:
# ============================================================
# BLOQUE 7: ESTRUCTURACIÓN (versión final, con lista blanca de secciones)
# ============================================================

# --- Parámetros ---
CONFIG["max_longitud_encabezado"] = 60
CONFIG["cierres_de_parrafo"] = (".", "?", "”", '"', ":")
CONFIG["similitud_encabezado_conocido"] = 0.80

# --- Lista blanca de secciones conocidas (ampliable con otros tipos de normativas) ---
SECCIONES_CONOCIDAS = [
    "CONSULTA",
    "RESPUESTA",
    "CONCLUSIÓN",
    "CONCLUSIONES",
    "TERRITORIALIDAD DEL SERVICIO",
    "ACTIVIDAD GRAVADA",
    "HECHOS",
    "CONSIDERACIONES",
    "NORMATIVIDAD APLICABLE",
    "MARCO NORMATIVO",
    "ANÁLISIS",
    "FUNDAMENTOS DE DERECHO",
]


def encabezado_conocido(linea: str) -> str | None:
    """Compara la línea contra la lista blanca, con tolerancia a variaciones de OCR."""
    candidato = linea.strip().rstrip(":").upper()
    if not candidato or len(candidato) > CONFIG["max_longitud_encabezado"]:
        return None
    for seccion in SECCIONES_CONOCIDAS:
        if SequenceMatcher(None, candidato, seccion).ratio() >= CONFIG["similitud_encabezado_conocido"]:
            return seccion
    return None


def es_posible_encabezado_generico(linea: str) -> bool:
    """Heurística de respaldo (no cambia la sección, solo se registra como candidato)."""
    linea = linea.strip()
    if not linea or len(linea) > CONFIG["max_longitud_encabezado"]:
        return False
    contenido = linea.rstrip(":")
    if not re.fullmatch(r"[A-ZÁÉÍÓÚÑ][A-ZÁÉÍÓÚÑ\s\-]*", contenido):
        return False
    letras = [c for c in contenido if c.isalpha()]
    return len(letras) >= 4


def detectar_articulo_citado(texto_parrafo: str) -> str | None:
    texto_inicio = texto_parrafo.strip()[:20]
    patron_completo = re.search(r"art[íi]culo\s+(\d+)[°º]?\.?\s*([A-ZÁÉÍÓÚÑ][^.]{0,60}\.)", texto_inicio, re.IGNORECASE)
    if patron_completo:
        return f"Artículo {patron_completo.group(1)}"
    patron_alterno = re.match(r"^(\d{1,3})\.\s*[A-ZÁÉÍÓÚÑ]", texto_inicio)
    if patron_alterno:
        return f"Artículo {patron_alterno.group(1)} (inferido, sin palabra 'Artículo' en OCR)"
    patron_mencion = re.search(r"art[íi]culo\s+(\d+)[°º]?", texto_parrafo, re.IGNORECASE)
    if patron_mencion:
        return f"Artículo {patron_mencion.group(1)} (mención, no cita textual)"
    return None


def segmentar_pagina_en_parrafos(texto_limpio: str, seccion_inicial: str) -> tuple[list[dict], str]:
    parrafos = []
    seccion_actual = seccion_inicial
    buffer_lineas = []
    encabezado_no_reconocido_pendiente = None

    def cerrar_parrafo_si_hay_contenido():
        nonlocal encabezado_no_reconocido_pendiente
        if buffer_lineas:
            texto_parrafo = " ".join(buffer_lineas).strip()
            if texto_parrafo:
                parrafos.append({
                    "seccion_documento": seccion_actual,
                    "texto_parrafo": texto_parrafo,
                    "articulo_citado": detectar_articulo_citado(texto_parrafo),
                    "posible_encabezado_no_reconocido": encabezado_no_reconocido_pendiente,
                })
            buffer_lineas.clear()
        encabezado_no_reconocido_pendiente = None

    for linea in texto_limpio.split("\n"):
        linea = linea.strip()
        if not linea:
            continue

        seccion_match = encabezado_conocido(linea)
        if seccion_match:
            cerrar_parrafo_si_hay_contenido()
            seccion_actual = seccion_match
            continue

        if es_posible_encabezado_generico(linea):
            encabezado_no_reconocido_pendiente = linea
            continue

        buffer_lineas.append(linea)
        if linea.rstrip().endswith(CONFIG["cierres_de_parrafo"]):
            cerrar_parrafo_si_hay_contenido()

    cerrar_parrafo_si_hay_contenido()
    return parrafos, seccion_actual


def estructurar_documento(df_paginas_doc: pd.DataFrame) -> pd.DataFrame:
    filas_estructuradas = []
    seccion_actual = "contenido_general"

    for _, fila in df_paginas_doc.sort_values("pagina").iterrows():
        parrafos_pagina, seccion_actual = segmentar_pagina_en_parrafos(fila["texto_limpio"], seccion_actual)
        for parrafo in parrafos_pagina:
            filas_estructuradas.append({
                "archivo": fila["archivo"],
                "pagina": fila["pagina"],
                **parrafo,
            })

    return pd.DataFrame(filas_estructuradas)


# --- Ejecutar sobre todos los documentos ---
partes_estructuradas = []
for nombre_archivo in df_limpio["archivo"].unique():
    df_doc = df_limpio[df_limpio["archivo"] == nombre_archivo]
    partes_estructuradas.append(estructurar_documento(df_doc))

df_estructurado = pd.concat(partes_estructuradas, ignore_index=True)
logger.info(f"Estructuración completada: {len(df_estructurado)} párrafo(s) detectado(s).")

print("Secciones detectadas:")
display(df_estructurado["seccion_documento"].value_counts())

2026-07-28 18:57:17,612 | INFO | Estructuración completada: 54 párrafo(s) detectado(s).
INFO:pipeline_normativas:Estructuración completada: 54 párrafo(s) detectado(s).


Secciones detectadas:


,count
seccion_documento,
ACTIVIDAD GRAVADA,19
RESPUESTA,18
TERRITORIALIDAD DEL SERVICIO,6
CONCLUSIÓN,6
CONSULTA,4
contenido_general,1


In [ ]:
pd.set_option("display.max_colwidth", 120)
display(df_estructurado[df_estructurado["articulo_citado"].notna()][["pagina", "seccion_documento", "articulo_citado", "texto_parrafo"]])

,pagina,seccion_documento,articulo_citado,texto_parrafo
2,1,CONSULTA,"Artículo 31 (mención, no cita textual)","De conformidad con los literales e y del artículo 31 del Decreto Distrital 601 de 2014, corresponde a esta Subdirecc..."
5,1,RESPUESTA,"Artículo 32 (mención, no cita textual)","Para dar respuesta a su consulta, nos referimos en primer lugar a los elementos determinantes del Impuesto de Indust..."
6,1,RESPUESTA,"Artículo 32 (inferido, sin palabra 'Artículo' en OCR)",32. Hecho generador.
10,2,RESPUESTA,"Artículo 33 (inferido, sin palabra 'Artículo' en OCR)",33. Actividad industrial.
12,2,RESPUESTA,"Artículo 34 (mención, no cita textual)",Artículo 34. Actividad comercial.
14,2,RESPUESTA,"Artículo 35 (mención, no cita textual)",Artículo 35. Actividad de servicio.
16,2,RESPUESTA,"Artículo 41 (mención, no cita textual)","Por su parte, los sujetos pasivos del impuesto al tenor de lo señalado en el artículo 41 del Decreto Distrital 352 d..."
17,2,RESPUESTA,"Artículo 54 (mención, no cita textual)","Según lo preceptuado en el artículo 54 de la Ley 1430 de 2010, modificado por el articulo 177 de la Ley 1607 de 2012..."
20,2,RESPUESTA,"Artículo 42 (mención, no cita textual)",Artículo 42. Base gravable.
31,4,ACTIVIDAD GRAVADA,"Artículo 476 (mención, no cita textual)",qe a SL tos efectos determine el MINTIC. Si el servicio no cumple con los elementos necesarios para Identificar plen...


In [ ]:
# ============================================================
# PASO INTERMEDIO CORREGIDO: filtrar párrafos que son mayormente ruido OCR
# ============================================================

CONFIG["min_ratio_palabras_validas"] = 0.55
CONFIG["min_palabras_alfabeticas_reales"] = 4  # mínimo de palabras de texto real (no números) para no ser ruido


def es_palabra_valida(palabra: str) -> bool:
    limpia = palabra.strip(".,;:¿?¡!\"'()")
    if not limpia:
        return False
    if limpia.isdigit() or re.match(r"^\d", limpia):
        return True
    if len(limpia) < 2:
        return False
    return any(v in limpia.lower() for v in "aeiouáéíóú")


def es_palabra_alfabetica_real(palabra: str) -> bool:
    """Palabra de texto real: no es número, tiene al menos 3 letras y una vocal."""
    limpia = palabra.strip(".,;:¿?¡!\"'()%")
    if not limpia or limpia.isdigit() or re.match(r"^\d", limpia):
        return False
    letras = [c for c in limpia if c.isalpha()]
    if len(letras) < 3:
        return False
    return any(v in limpia.lower() for v in "aeiouáéíóú")


def calcular_ratio_palabras_validas(texto: str) -> float:
    palabras = texto.split()
    if not palabras:
        return 0.0
    validas = sum(1 for p in palabras if es_palabra_valida(p))
    return validas / len(palabras)


def contar_palabras_alfabeticas_reales(texto: str) -> int:
    return sum(1 for p in texto.split() if es_palabra_alfabetica_real(p))


def recortar_ruido_inicial(texto: str, max_tokens_revisar: int = 6) -> str:
    tokens = texto.split()
    inicio_real = 0
    for i, token in enumerate(tokens[:max_tokens_revisar]):
        if es_palabra_valida(token):
            inicio_real = i
            break
    else:
        inicio_real = 0
    return " ".join(tokens[inicio_real:])


df_estructurado["texto_parrafo"] = df_estructurado["texto_parrafo"].apply(recortar_ruido_inicial)
df_estructurado["ratio_palabras_validas"] = df_estructurado["texto_parrafo"].apply(calcular_ratio_palabras_validas)
df_estructurado["num_palabras_alfabeticas_reales"] = df_estructurado["texto_parrafo"].apply(contar_palabras_alfabeticas_reales)

parrafos_antes = len(df_estructurado)
df_estructurado = df_estructurado[
    (df_estructurado["ratio_palabras_validas"] >= CONFIG["min_ratio_palabras_validas"])
    & (df_estructurado["num_palabras_alfabeticas_reales"] >= CONFIG["min_palabras_alfabeticas_reales"])
].reset_index(drop=True)
parrafos_despues = len(df_estructurado)

print(f"Párrafos antes del filtro: {parrafos_antes}")
print(f"Párrafos después del filtro: {parrafos_despues}")

pd.set_option("display.max_colwidth", 120)
display(df_estructurado[["pagina", "seccion_documento", "texto_parrafo", "ratio_palabras_validas", "num_palabras_alfabeticas_reales"]])

Párrafos antes del filtro: 54
Párrafos después del filtro: 47


,pagina,seccion_documento,texto_parrafo,ratio_palabras_validas,num_palabras_alfabeticas_reales
0,1,contenido_general,"AAA PECRETAMÍA 0 1644 HE a Sd 87 SUBDIRECCIÓN ACCOUNTER LTDA/DIAN MCcC Bogotá D.C, 4 de abril de 2019 Señores NIt_ 9...",0.842105,28
1,1,CONSULTA,28 No 46 -97 Tel : 4432300 Bogotá Referencia Radicado 2019ER32083 del 22/03/2019 jo Eb: Tema : ICA Subtema Territori...,0.903226,19
2,1,CONSULTA,"De conformidad con los literales e y del artículo 31 del Decreto Distrital 601 de 2014, corresponde a esta Subdirecc...",0.909836,85
3,1,CONSULTA,"¿Queremos establecer para efectos de la territorialidad del impuesto de industria y comercio, los servicios de cloud...",0.966667,20
4,1,CONSULTA,en el sitio donde se encuentran ubicados los servidores y se realiza efectivamente la actividad gravada?,0.937500,10
5,1,RESPUESTA,"Para dar respuesta a su consulta, nos referimos en primer lugar a los elementos determinantes del Impuesto de Indust...",0.918367,29
6,1,RESPUESTA,El hecho generador del Impuesto de industria y comercio está constituido por el ejercicio o realización directa o In...,0.890909,36
7,2,RESPUESTA,"uo) DEC TAMA As MALE De Igual manera cabe señalar, la definición de cada una de las actividades económicas descritas...",0.942857,21
8,2,RESPUESTA,"Es actividad industrial, la producción, extracción, fabricación, manufactura, confección, preparación, reparación, e...",0.935484,23
9,2,RESPUESTA,"Es actividad comercial, la destinada al expendio, compraventa o distribución de bienes y mercancías, tanto al por ma...",0.897959,33


In [ ]:
# ============================================================
# BLOQUE 8: EXPORTACIÓN (metadata + párrafos, listos para Sheets)
# ============================================================

import json
import unicodedata

def quitar_tildes(texto: str) -> str:
    texto_sin_tildes = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    return texto_sin_tildes.lower()


def extraer_metadata_documento(df_paginas_doc: pd.DataFrame) -> dict:
    texto_inicial = " ".join(df_paginas_doc.sort_values("pagina")["texto_limpio"].head(1))

    def buscar_campo(patron):
        match = re.search(patron, texto_inicial, re.IGNORECASE)
        return match.group(1).strip() if match else None

    return {
        "archivo": df_paginas_doc["archivo"].iloc[0],
        "radicado": buscar_campo(r"radicado\s+([\dA-Z]+)\s+del"),
        "fecha_respuesta": buscar_campo(r"(\d{1,2}\s+de\s+\w+\s+de\s+\d{4})"),
        "tema": buscar_campo(r"tema\s*:\s*([A-ZÁÉÍÓÚÑ]+)"),
        "subtema": buscar_campo(r"subtema\s*(.+?)(?:\n|respetada|$)"),
    }


metadata_documentos = []
for nombre_archivo in df_limpio["archivo"].unique():
    df_doc = df_limpio[df_limpio["archivo"] == nombre_archivo]
    metadata_documentos.append(extraer_metadata_documento(df_doc))

df_metadata = pd.DataFrame(metadata_documentos)
print("Metadata extraída por documento:")
display(df_metadata)

df_export = df_estructurado.copy()
df_export["orden_documento"] = df_export.groupby("archivo").cumcount()
df_export["id_parrafo"] = (
    df_export["archivo"].str.replace(".pdf", "", regex=False)
    + "_p" + df_export["orden_documento"].astype(str)
)
df_export["texto_busqueda"] = df_export["texto_parrafo"].apply(quitar_tildes)

columnas_finales = [
    "id_parrafo", "archivo", "pagina", "orden_documento",
    "seccion_documento", "articulo_citado", "texto_parrafo", "texto_busqueda",
]
df_export = df_export[columnas_finales]

print("\nTabla final de párrafos, lista para exportar:")
display(df_export.head(10))

ruta_csv_parrafos = PATHS["output"] / "parrafos_estructurados.csv"
ruta_csv_metadata = PATHS["output"] / "metadata_documentos.csv"
ruta_json_parrafos = PATHS["output"] / "parrafos_estructurados.json"

df_export.to_csv(ruta_csv_parrafos, index=False, encoding=CONFIG["encoding"])
df_metadata.to_csv(ruta_csv_metadata, index=False, encoding=CONFIG["encoding"])

with open(ruta_json_parrafos, "w", encoding=CONFIG["encoding"]) as f:
    json.dump(df_export.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

logger.info(f"Exportación completada: {len(df_export)} párrafos, {len(df_metadata)} documento(s).")
print(f"\n✅ Archivos generados en {PATHS['output']}:")
print(f"  - {ruta_csv_parrafos.name}")
print(f"  - {ruta_csv_metadata.name}")
print(f"  - {ruta_json_parrafos.name}")

Metadata extraída por documento:


,archivo,radicado,fecha_respuesta,tema,subtema
0,NuevoDocumento-2019-06-26-07.57.04.pdf,2019ER32083,4 de abril de 2019,ICA,Territorialidad actividad de servicio (Cloud computing/provider)



Tabla final de párrafos, lista para exportar:


,id_parrafo,archivo,pagina,orden_documento,seccion_documento,articulo_citado,texto_parrafo,texto_busqueda
0,NuevoDocumento-2019-06-26-07.57.04_p0,NuevoDocumento-2019-06-26-07.57.04.pdf,1,0,contenido_general,None,"AAA PECRETAMÍA 0 1644 HE a Sd 87 SUBDIRECCIÓN ACCOUNTER LTDA/DIAN MCcC Bogotá D.C, 4 de abril de 2019 Señores NIt_ 9...","aaa pecretamia 0 1644 he a sd 87 subdireccion accounter ltda/dian mccc bogota d.c, 4 de abril de 2019 senores nit_ 9..."
1,NuevoDocumento-2019-06-26-07.57.04_p1,NuevoDocumento-2019-06-26-07.57.04.pdf,1,1,CONSULTA,None,28 No 46 -97 Tel : 4432300 Bogotá Referencia Radicado 2019ER32083 del 22/03/2019 jo Eb: Tema : ICA Subtema Territori...,28 no 46 -97 tel : 4432300 bogota referencia radicado 2019er32083 del 22/03/2019 jo eb: tema : ica subtema territori...
2,NuevoDocumento-2019-06-26-07.57.04_p2,NuevoDocumento-2019-06-26-07.57.04.pdf,1,2,CONSULTA,"Artículo 31 (mención, no cita textual)","De conformidad con los literales e y del artículo 31 del Decreto Distrital 601 de 2014, corresponde a esta Subdirecc...","de conformidad con los literales e y del articulo 31 del decreto distrital 601 de 2014, corresponde a esta subdirecc..."
3,NuevoDocumento-2019-06-26-07.57.04_p3,NuevoDocumento-2019-06-26-07.57.04.pdf,1,3,CONSULTA,None,"¿Queremos establecer para efectos de la territorialidad del impuesto de industria y comercio, los servicios de cloud...","queremos establecer para efectos de la territorialidad del impuesto de industria y comercio, los servicios de cloud ..."
4,NuevoDocumento-2019-06-26-07.57.04_p4,NuevoDocumento-2019-06-26-07.57.04.pdf,1,4,CONSULTA,None,en el sitio donde se encuentran ubicados los servidores y se realiza efectivamente la actividad gravada?,en el sitio donde se encuentran ubicados los servidores y se realiza efectivamente la actividad gravada?
5,NuevoDocumento-2019-06-26-07.57.04_p5,NuevoDocumento-2019-06-26-07.57.04.pdf,1,5,RESPUESTA,"Artículo 32 (mención, no cita textual)","Para dar respuesta a su consulta, nos referimos en primer lugar a los elementos determinantes del Impuesto de Indust...","para dar respuesta a su consulta, nos referimos en primer lugar a los elementos determinantes del impuesto de indust..."
6,NuevoDocumento-2019-06-26-07.57.04_p6,NuevoDocumento-2019-06-26-07.57.04.pdf,1,6,RESPUESTA,None,El hecho generador del Impuesto de industria y comercio está constituido por el ejercicio o realización directa o In...,el hecho generador del impuesto de industria y comercio esta constituido por el ejercicio o realizacion directa o in...
7,NuevoDocumento-2019-06-26-07.57.04_p7,NuevoDocumento-2019-06-26-07.57.04.pdf,2,7,RESPUESTA,None,"uo) DEC TAMA As MALE De Igual manera cabe señalar, la definición de cada una de las actividades económicas descritas...","uo) dec tama as male de igual manera cabe senalar, la definicion de cada una de las actividades economicas descritas..."
8,NuevoDocumento-2019-06-26-07.57.04_p8,NuevoDocumento-2019-06-26-07.57.04.pdf,2,8,RESPUESTA,None,"Es actividad industrial, la producción, extracción, fabricación, manufactura, confección, preparación, reparación, e...","es actividad industrial, la produccion, extraccion, fabricacion, manufactura, confeccion, preparacion, reparacion, e..."
9,NuevoDocumento-2019-06-26-07.57.04_p9,NuevoDocumento-2019-06-26-07.57.04.pdf,2,9,RESPUESTA,None,"Es actividad comercial, la destinada al expendio, compraventa o distribución de bienes y mercancías, tanto al por ma...","es actividad comercial, la destinada al expendio, compraventa o distribucion de bienes y mercancias, tanto al por ma..."


2026-07-28 19:55:11,858 | INFO | Exportación completada: 47 párrafos, 1 documento(s).
INFO:pipeline_normativas:Exportación completada: 47 párrafos, 1 documento(s).



✅ Archivos generados en /content/drive/MyDrive/DAVIVIENDA/output:
  - parrafos_estructurados.csv
  - metadata_documentos.csv
  - parrafos_estructurados.json


In [ ]:
print("Metadata extraída por documento:")
display(df_metadata)

Metadata extraída por documento:


,archivo,radicado,fecha_respuesta,tema,subtema
0,NuevoDocumento-2019-06-26-07.57.04.pdf,2019ER32083,4 de abril de 2019,ICA,Territorialidad actividad de servicio (Cloud computing/provider)
